## Problem 1: In the lecture note, and also in Suton, the TD($\lambda$) algorithm is presented as

**TD($\lambda$) algorithm 1**:

- Initialize $V(s)$ arbitarily (but make sure $V(s) = 0$ if $s$ is terminal)
- Repeat for each episode
   - Initialize $E(s) = 0$ for all state $s\in \mathcal{S}$, and randomly choose a starting state $S$
   - While $S$ is not terminal:
      - Select action $A$ per policy $\pi(a|S)$ and take it
      - Observe reward $R$ and next state $S'$
      - Compute TD error: $\delta = R + \gamma V(S') - V(S)$
      - Increment eligible trace:  $E(S) = E(S) + 1$
      - For $s\in \mathcal{S}$:             **<== this is done for all states**
          - Update state value: $V(s) = V(s) + \alpha E(s) \delta $    **<== taking credit even if $s\ne S$**
          - Update the eligibility trace: $E(s) = \gamma\lambda E(s)$
      - $S = S'$
      
 
 


 **Yet the eligibility trace update is presented in the lecture note as**
 
 - Each time step, at state $S_t$, for **every state** $s$:
  - if $s\ne S_t$, discount this factor by $\gamma\lambda$
  $$E(s) = \gamma\lambda E(s),\ \text{ if } s \ne S_t$$
  - else we do the same discount for past $S_t$, but take full credit for the current $S_t$:
  $$E(s) = \gamma\lambda E(s) + 1,\ \text{ if } s = S_t$$
      

**Using this eligibility trace update logic, the TD($\lambda$) algorithm should be**

**TD($\lambda$) algorithm 2**:

- Initialize $V(s)$ arbitarily (but make sure $V(s) = 0$ if $s$ is terminal)
- Repeat for each episode
   - Initialize $E(s) = 0$ for all state $s\in \mathcal{S}$, and randomly choose a starting state $S$
   - While $S$ is not terminal:
      - Select action $A$ per policy $\pi(a|S)$ and take it
      - Observe reward $R$ and next state $S'$
      - Compute TD error: $\delta = R + \gamma V(S') - V(S)$
      - For $s\in \mathcal{S}$:             **<== this is done for all states** different part
          - Update the eligibility trace: $E(s) = \gamma\lambda E(s)$
      - Increment eligible trace:  $E(S) = E(S) + 1$
      - For $s\in \mathcal{S}$:             **<== this is done for all states**
          - Update state value: $V(s) = V(s) + \alpha E(s) \delta $    **<== taking credit even if $s\ne S$**
      - $S = S'$
      
 


## Question: Are Algorithms 1 and 2 same or different?

**Task 1 (30pts)**:

- If you think they are the same, explain why so, and implement both on a small MPD example to demonstrate the results
- If you think they are different, give a simple example to demonstrate the difference

**Task 2 (30pts)**:
- Try ask this question to an LLM, by copying the two pseudo code into an LLM of your choice. What does the LLM tell you? If your LLM disagree with you, reflect on the difference

**Task 3 (40pts)**: implement these two algorithms by completing the following code, and check the results on the SevenState problem in the lecture note, are they the same?

In [3]:
import random
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
class SevenStateEnv(gym.Env):
    def __init__(self, start_state = 3):
        self.agent_state = start_state
        self.action_space = [-1, 1] # left or right
        self.goal_states = {0, 6}
        self.states = [0,1,2,3,4,5,6]
        self.start_state = start_state
    
    def policy(self, s): # return [[a, p], ...] for all a in action space, where p is the probability of taking action a in state s
        if s >= 1 and s <= 5:
            return [[-1, 0.5],[1,0.5]]
        else:
            raise(ValueError(f"not a normal state {s}"))
        
    def dynamics(self, s, a): #this calculates P(s', r|s, a), it return [[s', r, p],...]  
        if s >= 1 and s <= 4:
            return[[s+a, 0, 1.0]]
            
        if s == 5:
            if a == -1:
                return [[4, 0, 1.0]]
            else:
                return [[6, 1, 1.0]]
        raise(ValueError(f"querying dynamics for non-normal state {s}"))
             
    def reset(self, options = {}): # return initial observation, info. Reset the start_state if options has "start_state"
        if "start_state" in options:
            self.start_state = options["start_state"]
        self.agent_state = self.start_state
        return self.agent_state, {}
    
    def sample_action(self, s): # return a sampled action according to the policy in state s
        pi = self.policy(s)
        actions = [x[0] for x in pi]
        probs = [x[1] for x in pi]
        return random.choices(actions, weights=probs, k=1)[0]
        
    def sample_state_reward(self, s, a): # given action, find the next state and reward based on dynamics
        res = self.dynamics(s, a) # [[s', r, p],...]
        states_rewards = [srp[:2] for srp in res]
        probs = [srp[-1] for srp in res]
        return random.choices(states_rewards, weights=probs, k=1)[0]
        
    def step(self, a): # return observation, reward, terminated, truncated, info
        s = self.agent_state
        [s2, reward] = self.sample_state_reward(s, a) # sample s', r given s, a
        self.agent_state = s2
        done = self.agent_state in self.goal_states
        return self.agent_state, reward, done, False, {}
        
    def episode(self, start_state = None): # return an episode [(s, a, r, s'), ...]
        res = []
        if start_state is not None:
            self.reset(options={"start_state": start_state})
        state = self.start_state
        while True:
            action = self.sample_action(state)
            obs, reward, terminated, truncated, info = self.step(action)
            new_state = obs
            res.append((state, action, reward, new_state))
            state = new_state
            if terminated:
                self.reset()
                return res

In [4]:
import random
def TDLambda_ValuePrediction_test(env, gamma = 1, alpha = 0.1, lamb = 0.5, max_iter = 1000, seed = 123, v_default = 0.5, schema = 'alg1'):
    random.seed(seed)
    states = env.states
    v_t = {s:[] for s in states} # this is used to record progress of v(s) over time
    v = {s:v_default for s in states}
    terminal_states = env.goal_states
    
    for s in states:
        if s in terminal_states:
            v[s] = 0.0 # terminal state much have 0 value
            
    for i in range(max_iter):
        s0 = random.choice([s for s in states if s not in terminal_states])
        seq = env.episode(s0) # seq = [(s, a, r, s'),...]
        E = {s:0 for s in states} # eligibility trace
        for (s, a, r, s2) in seq:
            delta = r + gamma*v[s2] - v[s]
            if schema == 'alg1':
                E[s] += 1
                for s_all in states:
                    v[s_all] += alpha * delta * E[s_all]
                    E[s_all] *= gamma * lamb

            else:
                for s_all in states:
                    E[s_all] *= gamma * lamb
                    if s_all == s:
                        E[s_all] += 1
                    v[s_all] += alpha * delta * E[s_all]
        for s in states:
            v_t[s] = v_t[s] + [v[s]]
    return v, v_t

In [5]:
env = SevenStateEnv()
TDLambda_ValuePrediction_test(env, schema = 'alg1')[0], TDLambda_ValuePrediction_test(env, schema = 'alg2')[0]

({0: 0.0,
  1: 0.12362269235715764,
  2: 0.3790266393999385,
  3: 0.5432292221685533,
  4: 0.7653411980507878,
  5: 0.8702536908061262,
  6: 0.0},
 {0: 0.0,
  1: 0.12362269235715764,
  2: 0.3790266393999385,
  3: 0.5432292221685533,
  4: 0.7653411980507878,
  5: 0.8702536908061262,
  6: 0.0})

# 1:
For the context of the stationary situation, I think they are the same eventually. At the end of each turn, the E(s) for each s is the same. Even though they update V(s) before and after the decay of E(s), but the decay term will be vanished after many turns and only remain the true answer.

# 2:
Claude: Used around 10s to find the answer and gaves out the equation rough statement, even though I firstly thought they are different. But it's unfair since I told Claude a lot of times how to generate the good answers and it also kept my instructions in memory.

GPT: Fastest for I didn't open the thinking mode. But unfortunately the equations are not in markdown readable mode.

Gemini: Thought they are different only based on the part they do decay.

Deepseek:Generate a lot of words but still claim they are different. Just basically made what Gemini said longer without focusing on the important parts.

Kimi(fast mode): Don't even say they are the same or different, only after inquiring I got the correct answer.



# 3:
As the number showed above,they are the same.

In [6]:
! python -m jupyter nbconvert 07_Model_Free_Prediction_HW_TD_lambda-1.ipynb --to webpdf


[NbConvertApp] Converting notebook 07_Model_Free_Prediction_HW_TD_lambda-1.ipynb to webpdf
[NbConvertApp] Building PDF
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 170630 bytes to 07_Model_Free_Prediction_HW_TD_lambda-1.pdf
